Python script to transform raw data

In [ ]:
use database healthpulse_manual_db

In [ ]:
create schema if not exists bronze_schema;
create schema if not exists silver_schema;

In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
# Fetch data from bronze table
df = session.table('bronze_schema."raw_data"')
df.show(5)

In [ ]:
# Store data in pandas dataframe
raw_data = df.to_pandas()

In [ ]:
# Check data type of every column
raw_data.dtypes

In [ ]:
# Check null value count in table
raw_data.isnull().sum()

In [ ]:
# Uniqueness Data Quality Check
# Check unique count for appointment_id
raw_data['appointment_id'].is_unique

In [ ]:
# Check categorical data quality
categorical_cols = ['insurance_type', 'specialty', 'clinic_assignment', 'clinic_name', 'city', 'is_no_show_0_1']
print("\nUnique values in categorical columns:")
for col in categorical_cols:
    unique_vals = raw_data[col].unique()
    print(col, unique_vals)

In [ ]:
# Rename is_no_show column 
raw_data.rename(columns={'is_no_show_0_1': 'is_no_show'}, inplace=True)

In [ ]:
# appointment_date datatype conversion
raw_data['appointment_date'].head(5)

In [ ]:
raw_data['appointment_date'] = pd.to_datetime(raw_data['appointment_date'], format = '%m/%d/%Y').dt.date


In [ ]:
raw_data['appointment_date'].dtype
print(type(raw_data['appointment_date'].iloc[0]))

In [ ]:
# appointment_time datattype conversion
raw_data['appointment_time'].head(5)

In [ ]:
raw_data['appointment_time'] = pd.to_datetime(raw_data['appointment_time'], format = '%I:%M:%S %p').dt.time

In [ ]:
raw_data['appointment_time'].dtype
print(type(raw_data['appointment_time'].iloc[0]))

In [ ]:
raw_data['lead_time_days'].dtype
raw_data['age'].dtype

In [ ]:
# status column creation based on is_no_show column
raw_data['status'] = raw_data['is_no_show'].map({0: 'Completed', 1: 'No Show'})

In [ ]:
# check data types
raw_data.dtypes

SQL script for creating silver schema and table

In [ ]:
-- create table to store transformed data
create or replace table silver_schema.silver_data (
appointment_id Varchar(20),
patient_id Varchar(20),
provider_id Varchar(20),
appointment_date date,
appointment_time time,
lead_time_days number(5),
wait_time_minutes number(5,2),
is_no_show number(2),
age number(3),
insurance_type Varchar(20),
specialty Varchar(100),
provider_clinic_id Varchar(20),
clinic_assignment Varchar(20),
clinic_name Varchar(100),
city Varchar(20),
hours Varchar(20),
status varchar(20)
)
      

In [ ]:
# rearrange columns as per silver table columns
raw_data = raw_data[['appointment_id', 'patient_id', 'provider_id',
                       'appointment_date', 'appointment_time', 'lead_time_days',
                       'wait_time_minutes', 'is_no_show', 'age',
                       'insurance_type', 'specialty', 'provider_clinic_id',
                       'clinic_assignment', 'clinic_name', 'city', 'hours', 'status']]

In [ ]:
# insert data into created table
try:
    # Convert your Pandas DataFrame to Snowpark DataFrame
    df_snowpark = session.create_dataframe(raw_data)

    # Append to the pre-created Silver table
    df_snowpark.write.save_as_table("silver_schema.silver_data", mode="append")
    
except Exception as e:
    print(e)